# 07 — Aggregator Internals Analysis
Visualises the defense pipeline internals: MAD score distributions, cosine similarity matrices, and simplex weight distributions.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.components.server.aggregator import (
    extract_final_layer, compute_layer_wise_cosine_similarity,
    compute_mad_scores, temperature_scaled_softmax, project_capped_simplex
)
from src.components.model.model import MLPClassifier, get_model_parameters
from src.configs.config import CONFIG

cfg = CONFIG['model']
n_clients = 20  # simulate a round with 20 clients
n_attackers = 6  # 30% of 20

print('Simulating aggregator internals with synthetic client updates...')

In [ ]:
# Generate synthetic benign + attacker updates
benign_updates = [get_model_parameters(MLPClassifier(cfg['input_dim'], cfg['hidden_dims'], cfg['num_classes'])) for _ in range(n_clients - n_attackers)]

# Attacker updates: scale final layer by large factor (simulate label-flip divergence)
attacker_updates = []
for _ in range(n_attackers):
    params = get_model_parameters(MLPClassifier(cfg['input_dim'], cfg['hidden_dims'], cfg['num_classes']))
    params[-2] = params[-2] * 8.0  # corrupt final weight layer
    attacker_updates.append(params)

all_updates = benign_updates + attacker_updates
labels = ['benign'] * (n_clients - n_attackers) + ['attacker'] * n_attackers
print(f'Generated {n_clients - n_attackers} benign + {n_attackers} attacker updates.')

In [ ]:
# Step 1: Extract final layers
final_layers = np.stack([extract_final_layer(nd) for nd in all_updates])
print(f'Final layer vectors shape: {final_layers.shape}')

# Step 2: Cosine similarity matrix
sim_matrix = compute_layer_wise_cosine_similarity(final_layers)

fig, ax = plt.subplots(figsize=(8, 7))
tick_labels = [f'B{i}' for i in range(n_clients - n_attackers)] + [f'A{i}' for i in range(n_attackers)]
sns.heatmap(sim_matrix, cmap='coolwarm', center=0, ax=ax,
            xticklabels=tick_labels, yticklabels=tick_labels, linewidths=0.3)
ax.set_title('Cosine Similarity Matrix (B=benign, A=attacker)')
plt.tight_layout()
plt.savefig(Path('..') / 'artifacts' / 'plots' / 'aggregator_cosine_sim.png', dpi=150)
plt.show()

In [ ]:
# Step 3: MAD Z-scores
mad_scores = compute_mad_scores(sim_matrix)
colors = ['tomato' if l == 'attacker' else 'steelblue' for l in labels]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(n_clients), mad_scores, color=colors, edgecolor='white')
ax.axhline(CONFIG['defense']['mad_threshold'], color='red', linestyle='--', label=f'MAD threshold ({CONFIG["defense"]["mad_threshold"]})')
ax.set_xticks(range(n_clients))
ax.set_xticklabels(tick_labels, rotation=45)
ax.set_ylabel('MAD Z-Score')
ax.set_title('MAD Anomaly Scores (red = attacker, blue = benign)')
ax.legend()
plt.tight_layout()
plt.savefig(Path('..') / 'artifacts' / 'plots' / 'aggregator_mad_scores.png', dpi=150)
plt.show()
print(f'Flagged as anomalous (score < {CONFIG["defense"]["mad_threshold"]}): {(mad_scores < CONFIG["defense"]["mad_threshold"]).sum()}/{n_clients}')

In [ ]:
# Step 4: Temperature-scaled softmax
trust_weights = temperature_scaled_softmax(mad_scores, CONFIG['defense']['temperature'])

# Step 5: Capped simplex projection
cap_t = 1.0 / (n_clients - int(0.3 * n_clients))
final_weights = project_capped_simplex(trust_weights, cap_t)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(n_clients), trust_weights, color=colors, edgecolor='white')
axes[0].set_title('Softmax Trust Weights (before simplex projection)')
axes[0].set_ylabel('Weight')
axes[0].axhline(cap_t, color='red', linestyle='--', label=f'cap_t={cap_t:.3f}')
axes[0].legend()

axes[1].bar(range(n_clients), final_weights, color=colors, edgecolor='white')
axes[1].set_title('Final Aggregation Weights (after capped simplex)')
axes[1].set_ylabel('Weight')
axes[1].axhline(cap_t, color='red', linestyle='--', label=f'cap_t={cap_t:.3f}')
axes[1].legend()

for ax in axes:
    ax.set_xticks(range(n_clients))
    ax.set_xticklabels(tick_labels, rotation=45, fontsize=7)

plt.suptitle('Defense Pipeline — Trust Weights (red=attacker, blue=benign)')
plt.tight_layout()
plt.savefig(Path('..') / 'artifacts' / 'plots' / 'aggregator_weights.png', dpi=150)
plt.show()
print(f'Attackers with zero weight: {(final_weights[n_clients-n_attackers:] == 0).sum()}/{n_attackers}')